In [9]:
# Used to generate input for lollipopPlot by using ProteinPaint
import glob, os
import pandas as pd

OUTDIR = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/20251224/Batch2_021326/PCGR_021326/MAF/proteinpaint_inputs"
os.makedirs(OUTDIR, exist_ok=True)

MAFDIR = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/20251224/Batch2_021326/PCGR_021326/MAF"

def map_class(vc):
    vc = ("" if vc is None else str(vc))

    if "Missense" in vc:
        return "M"          # MISSENSE
    if "Nonsense" in vc:
        return "N"          # NONSENSE
    if "Frame_Shift" in vc:
        return "F"          # FRAMESHIFT
    if "In_Frame_Del" in vc:
        return "D"          # PROTEINDEL
    if "In_Frame_Ins" in vc:
        return "I"          # PROTEININS
    if "Splice" in vc:
        return "L"          # SPLICE
    if "SPLICE_REGION" in vc:
        return "P"          # SPLICE_REGION
    if "Silent" in vc:
        return "S"          # SILENT

    return "ProteinAltering"

dfs = []
for maf in glob.glob(os.path.join(MAFDIR, "*.maf")):
    sample = os.path.basename(maf).replace(".pcgr.grch37.maf", "").replace(".maf", "")
    df = pd.read_csv(maf, sep="\t", dtype=str, comment="#", low_memory=False)

    # keep only relevant variant classes
    vc = df["Variant_Classification"].fillna("")
    df = df[vc.str.contains("Missense|Nonsense|Frame_Shift|In_Frame|Splice", regex=True)]
    if df.empty:
        continue

    # protein change -> mutation name and AA position
    df["mutation_name"] = df["HGVSp_Short"].fillna("").str.replace(r"^p\.", "", regex=True)
    df["position"] = df["mutation_name"].str.extract(r"(\d+)")
    df["class"] = df["Variant_Classification"].apply(map_class)

    df["sample"] = sample
    df["gene"] = df["Hugo_Symbol"].fillna("")

    # keep only rows that have gene + AA position (ProteinPaint lollipop needs position)
    df = df[(df["gene"] != "") & df["position"].notna()]
    if df.empty:
        continue

    dfs.append(df[["gene", "mutation_name", "position", "class", "sample"]])

if not dfs:
    raise SystemExit("No qualifying variants found in any MAF files.")

allc = pd.concat(dfs, ignore_index=True)

GENE_OUT = os.path.join(OUTDIR, "by_gene")
os.makedirs(GENE_OUT, exist_ok=True)

for gene in ["ESR1", "ERBB2", "TP53", "PIK3CA", "GATA3", "FOXA1", "CDH1"]:
    gdf = allc[allc["gene"] == gene].copy()
    out = os.path.join(GENE_OUT, f"ALLSAMPLES_{gene}.proteinpaint.tsv")
    gdf[["mutation_name", "position", "class", "sample"]].to_csv(out, sep="\t", index=False)
    print(gene, len(gdf))


ESR1 11
ERBB2 17
TP53 5
PIK3CA 16
GATA3 4
FOXA1 4
CDH1 5


In [8]:
# Used to generate input for lollipopPlot by using ProteinPaint
import glob, os
import pandas as pd

OUTDIR = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/PCGR/PCGR_010526/MAF/proteinpaint_inputs"
os.makedirs(OUTDIR, exist_ok=True)

MAFDIR = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/PCGR/PCGR_010526/MAF"

def map_class(vc):
    vc = ("" if vc is None else str(vc))

    if "Missense" in vc:
        return "M"          # MISSENSE
    if "Nonsense" in vc:
        return "N"          # NONSENSE
    if "Frame_Shift" in vc:
        return "F"          # FRAMESHIFT
    if "In_Frame_Del" in vc:
        return "D"          # PROTEINDEL
    if "In_Frame_Ins" in vc:
        return "I"          # PROTEININS
    if "Splice" in vc:
        return "L"          # SPLICE
    if "SPLICE_REGION" in vc:
        return "P"          # SPLICE_REGION
    if "Silent" in vc:
        return "S"          # SILENT

    return "ProteinAltering"

dfs = []
for maf in glob.glob(os.path.join(MAFDIR, "*.maf")):
    sample = os.path.basename(maf).replace(".pcgr.grch37.maf", "").replace(".maf", "")
    df = pd.read_csv(maf, sep="\t", dtype=str, comment="#", low_memory=False)

    # keep only relevant variant classes
    vc = df["Variant_Classification"].fillna("")
    df = df[vc.str.contains("Missense|Nonsense|Frame_Shift|In_Frame|Splice", regex=True)]
    if df.empty:
        continue

    # protein change -> mutation name and AA position
    df["mutation_name"] = df["HGVSp_Short"].fillna("").str.replace(r"^p\.", "", regex=True)
    df["position"] = df["mutation_name"].str.extract(r"(\d+)")
    df["class"] = df["Variant_Classification"].apply(map_class)

    df["sample"] = sample
    df["gene"] = df["Hugo_Symbol"].fillna("")

    # keep only rows that have gene + AA position (ProteinPaint lollipop needs position)
    df = df[(df["gene"] != "") & df["position"].notna()]
    if df.empty:
        continue

    dfs.append(df[["gene", "mutation_name", "position", "class", "sample"]])

if not dfs:
    raise SystemExit("No qualifying variants found in any MAF files.")

allc = pd.concat(dfs, ignore_index=True)

GENE_OUT = os.path.join(OUTDIR, "by_gene")
os.makedirs(GENE_OUT, exist_ok=True)

for gene in ["ESR1", "ERBB2", "TP53", "PIK3CA", "GATA3", "FOXA1", "CDH1"]:
    gdf = allc[allc["gene"] == gene].copy()
    out = os.path.join(GENE_OUT, f"ALLSAMPLES_{gene}.proteinpaint.tsv")
    gdf[["mutation_name", "position", "class", "sample"]].to_csv(out, sep="\t", index=False)
    print(gene, len(gdf))


ESR1 3
ERBB2 8
TP53 5
PIK3CA 2
GATA3 2
FOXA1 0
CDH1 0


In [7]:
# Used to generate input for lollipopPlot by using ProteinPaint
import glob, os
import pandas as pd

OUTDIR = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/PCGR/PCGR_012425/MAF/proteinpaint_inputs"
os.makedirs(OUTDIR, exist_ok=True)

MAFDIR = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/EnRich/PCGR/PCGR_012425/MAF"

def map_class(vc):
    vc = ("" if vc is None else str(vc))

    if "Missense" in vc:
        return "M"          # MISSENSE
    if "Nonsense" in vc:
        return "N"          # NONSENSE
    if "Frame_Shift" in vc:
        return "F"          # FRAMESHIFT
    if "In_Frame_Del" in vc:
        return "D"          # PROTEINDEL
    if "In_Frame_Ins" in vc:
        return "I"          # PROTEININS
    if "Splice" in vc:
        return "L"          # SPLICE
    if "SPLICE_REGION" in vc:
        return "P"          # SPLICE_REGION
    if "Silent" in vc:
        return "S"          # SILENT

    return "ProteinAltering"

dfs = []
for maf in glob.glob(os.path.join(MAFDIR, "*.maf")):
    sample = os.path.basename(maf).replace(".pcgr.grch37.maf", "").replace(".maf", "")
    df = pd.read_csv(maf, sep="\t", dtype=str, comment="#", low_memory=False)

    # keep only relevant variant classes
    vc = df["Variant_Classification"].fillna("")
    df = df[vc.str.contains("Missense|Nonsense|Frame_Shift|In_Frame|Splice", regex=True)]
    if df.empty:
        continue

    # protein change -> mutation name and AA position
    df["mutation_name"] = df["HGVSp_Short"].fillna("").str.replace(r"^p\.", "", regex=True)
    df["position"] = df["mutation_name"].str.extract(r"(\d+)")
    df["class"] = df["Variant_Classification"].apply(map_class)

    df["sample"] = sample
    df["gene"] = df["Hugo_Symbol"].fillna("")

    # keep only rows that have gene + AA position (ProteinPaint lollipop needs position)
    df = df[(df["gene"] != "") & df["position"].notna()]
    if df.empty:
        continue

    dfs.append(df[["gene", "mutation_name", "position", "class", "sample"]])

if not dfs:
    raise SystemExit("No qualifying variants found in any MAF files.")

allc = pd.concat(dfs, ignore_index=True)

GENE_OUT = os.path.join(OUTDIR, "by_gene")
os.makedirs(GENE_OUT, exist_ok=True)

for gene in ["ESR1", "ERBB2", "TP53", "PIK3CA", "GATA3", "FOXA1", "CDH1"]:
    gdf = allc[allc["gene"] == gene].copy()
    out = os.path.join(GENE_OUT, f"ALLSAMPLES_{gene}.proteinpaint.tsv")
    gdf[["mutation_name", "position", "class", "sample"]].to_csv(out, sep="\t", index=False)
    print(gene, len(gdf))


ESR1 2
ERBB2 0
TP53 0
PIK3CA 0
GATA3 0
FOXA1 0
CDH1 0
